# Step 10: Skill 시스템 — 프롬프트 기반 워크플로우 조합

## 학습 목표

- **Tool(기능 단위) vs Skill(워크플로우 단위)**의 차이를 이해합니다
- **inline**(현재 대화에 프롬프트 주입) vs **fork**(서브에이전트로 독립 실행) 실행 방식을 비교합니다
- **마크다운 + YAML frontmatter**로 사용자 정의 스킬을 작성하는 방법을 배웁니다

> **Tool vs Skill**
> Tool은 "파일 읽기", "명령어 실행"같은 **기능 단위**입니다.
> Skill은 "커밋하기", "코드 리뷰"같은 **워크플로우 단위**로, 프롬프트 + 도구 제한의 조합입니다.
> 즉, Skill = 프롬프트 템플릿 + allowed_tools. 코드 한 줄 없이 마크다운만으로 새 기능을 추가할 수 있습니다.

---

## Claude Code 소스 분석

### 10-1. PromptCommand — 스킬의 타입 정의 (src/types/command.ts)

Claude Code에서 스킬은 `PromptCommand`라는 타입으로 정의됩니다. 핵심은 `context` 필드입니다:

```typescript
// src/types/command.ts

type PromptCommand = {
  name: string                          // 스킬 이름 (예: "commit", "review-pr")
  description: string                   // 스킬 설명
  content: string                       // 프롬프트 템플릿 (마크다운 본문)
  context: 'inline' | 'fork'           // ★ 실행 모드
  allowedTools?: string[]              // 사용 가능한 도구 제한
  whenToUse?: string                   // LLM에게 언제 사용할지 알려주는 힌트
}
```

**두 가지 실행 모드:**
- **`inline`**: 프롬프트를 현재 대화에 주입합니다. 빠르고 컨텍스트를 공유하지만, 현재 대화를 "오염"시킵니다.
- **`fork`**: 새 서브에이전트를 생성하여 독립 실행합니다. 격리되지만, 별도 API 호출이 필요합니다.

### 10-2. BundledSkillDefinition (src/skills/bundledSkills.ts)

Claude Code에는 `commit`, `review-pr`, `simplify` 등의 내장 스킬이 있습니다:

```typescript
// src/skills/bundledSkills.ts

interface BundledSkillDefinition {
  name: string
  instructions: string               // 스킬의 프롬프트
  trigger: string                     // "사용자가 /commit을 입력할 때"
  allowedTools?: string[]
}

const bundledSkills: BundledSkillDefinition[] = [
  {
    name: "commit",
    instructions: `현재 Git 저장소의 변경사항을 분석하고 커밋...`,
    trigger: "When the user asks to commit changes",
    allowedTools: ["Bash"],           // ★ Bash만 사용 가능하도록 제한
  },
  // ...
]
```

**설계 포인트:** `allowedTools`로 스킬이 사용할 수 있는 도구를 제한합니다. 예를 들어 `commit` 스킬은 `Bash`만 사용할 수 있고, `Read`나 `Write`는 사용하지 못합니다. 이것이 "Skill = prompt + tool restrictions"의 핵심입니다.

### 10-3. SkillTool — inline vs fork 실행 경로 (src/tools/SkillTool/SkillTool.ts)

```typescript
// src/tools/SkillTool/SkillTool.ts (핵심 실행 로직 간소화)

async function executeSkill(skill: PromptCommand, args: string) {
  // 프롬프트 렌더링
  const expandedPrompt = skill.content.replace("{args}", args)

  if (skill.context === 'inline') {
    // ★ inline: 프롬프트를 현재 대화에 주입
    // 새 user 메시지로 프롬프트를 추가하면,
    // LLM이 이 지시를 읽고 현재 대화 흐름 안에서 수행합니다.
    return injectAsUserMessage(expandedPrompt, skill.allowedTools)
  }

  if (skill.context === 'fork') {
    // ★ fork: 서브에이전트를 생성하여 독립 실행
    // AgentTool을 통해 새 에이전트 루프를 시작합니다.
    // 부모 대화의 컨텍스트는 상속하지만, 독립된 메시지 히스토리를 가집니다.
    return await forkSubagent(expandedPrompt, skill.allowedTools)
  }
}
```

### 10-4. 마크다운 스킬 로딩 (src/skills/loadSkillsDir.ts)

사용자가 `.claude/commands/` 디렉토리에 `.md` 파일을 두면 자동으로 커스텀 스킬로 로드됩니다:

```typescript
// src/skills/loadSkillsDir.ts (핵심 로직 간소화)

function loadSkillsFromDir(dir: string): PromptCommand[] {
  const files = fs.readdirSync(dir).filter(f => f.endsWith('.md'))
  return files.map(file => {
    const content = fs.readFileSync(path.join(dir, file), 'utf-8')
    const { frontmatter, body } = parseFrontmatter(content)
    return {
      name: frontmatter.name ?? path.basename(file, '.md'),
      description: frontmatter.description ?? '',
      content: body,                     // 마크다운 본문 = 프롬프트
      context: frontmatter.context ?? 'inline',
      allowedTools: frontmatter.allowed_tools,
    }
  })
}
```

**핵심 인사이트:** 스킬은 코드가 아니라 **프롬프트**입니다. 마크다운 파일 하나만 작성하면 새로운 워크플로우를 추가할 수 있습니다. 이것이 Claude Code의 확장성을 극대화하는 설계입니다.

---

## Python으로 구현하기

### 구현 1: SkillDefinition과 SkillRegistry

`mini_claude/skills/loader.py`에 구현된 스킬 시스템의 핵심 구조를 살펴봅니다.
Claude Code의 `PromptCommand`를 Python `dataclass`로 구현했습니다.

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.skills.loader import (
    SkillDefinition,
    SkillRegistry,
    BUNDLED_SKILLS,
    create_default_registry,
    load_skills_from_dir,
)

# --- 1. SkillDefinition 구조 확인 ---
# Claude Code의 PromptCommand에 대응합니다.

print("=== SkillDefinition 필드 ===")
sample = BUNDLED_SKILLS[0]
print(f"  name:            {sample.name}")
print(f"  description:     {sample.description}")
print(f"  context:         {sample.context}")        # 'inline' 또는 'fork'
print(f"  allowed_tools:   {sample.allowed_tools}")   # 도구 제한
print(f"  when_to_use:     {sample.when_to_use}")
print(f"  prompt_template: {sample.prompt_template[:80]}...")

print("\n=== 번들 스킬 목록 (Claude Code 내장 스킬에 대응) ===")
for skill in BUNDLED_SKILLS:
    mode_icon = "[inline]" if skill.context == "inline" else "[fork]  "
    tools = ", ".join(skill.allowed_tools) if skill.allowed_tools else "(모든 도구)"
    print(f"  {mode_icon} {skill.name:12s} — {skill.description[:40]}  도구: {tools}")

# --- 2. 프롬프트 렌더링 ---
print("\n=== 프롬프트 렌더링 (변수 치환) ===")
commit_skill = BUNDLED_SKILLS[0]
rendered = commit_skill.render(args="-m 'fix: 버그 수정'")
print(rendered)

### 구현 2: SkillRegistry와 SkillTool

`SkillRegistry`는 스킬을 등록하고 이름으로 조회하는 저장소입니다.
`SkillTool`은 LLM이 스킬을 호출할 때 사용하는 도구로, inline/fork 분기를 처리합니다.

In [ ]:
from mini_claude.tools.skill_tool import SkillTool, SkillExecutionResult

# --- 1. SkillRegistry 사용 ---
registry = create_default_registry()
print(f"등록된 스킬 수: {len(registry)}")
print(f"스킬 이름 목록: {registry.list_names()}")

# 이름으로 조회
review_skill = registry.find_by_name("review")
print(f"\n'review' 스킬 상세:")
print(f"  context: {review_skill.context}")    # fork — 서브에이전트로 독립 실행
print(f"  allowed_tools: {review_skill.allowed_tools}")  # None — 모든 도구 사용 가능

# --- 2. SkillTool — inline 실행 ---
skill_tool = SkillTool(registry)
print(f"\n=== SkillTool inline 실행 ===")

# 'commit' 스킬은 inline 모드 — 현재 대화에 프롬프트를 주입
result = await skill_tool.call({"skill": "commit", "args": "모든 변경사항"})
print(result.data[:200])

# --- 3. SkillTool — fork 실행 ---
print(f"\n=== SkillTool fork 실행 ===")

# 'review' 스킬은 fork 모드 — 서브에이전트가 독립적으로 처리
result = await skill_tool.call({"skill": "review", "args": "PR #42"})
print(result.data[:200])

# --- 4. execute_inline 편의 메서드 ---
print(f"\n=== execute_inline (대화 주입용) ===")
inline_result = await skill_tool.execute_inline("simplify", "utils.py")
if inline_result:
    print(f"  mode: {inline_result.mode}")
    print(f"  prompt: {inline_result.expanded_prompt[:100]}...")
    print(f"  context_messages: {len(inline_result.context_messages)}개")

---

## 실습: 커스텀 스킬을 마크다운 파일로 작성하고 로드하기

Claude Code에서 `.claude/commands/` 디렉토리에 `.md` 파일을 두면 커스텀 슬래시 커맨드가 됩니다.
같은 방식으로 마크다운 파일을 만들어 커스텀 스킬을 로드해봅시다.

In [ ]:
import tempfile
from pathlib import Path

# --- 1. 커스텀 스킬 디렉토리 생성 ---
skills_dir = Path(tempfile.mkdtemp(prefix="skills_"))

# --- 2. 마크다운 스킬 파일 작성 ---
# YAML frontmatter + 프롬프트 본문

# 스킬 1: 코드 설명 (inline 모드)
explain_skill = """\
---
name: explain
description: 코드를 읽고 한국어로 설명합니다
context: inline
allowed_tools: [Read, Grep, Glob]
when_to_use: 사용자가 코드 설명을 요청할 때
---
다음 코드를 분석하고 한국어로 설명해주세요.

설명에 포함할 내용:
1. 이 코드의 목적과 역할
2. 주요 클래스/함수의 동작 방식
3. 핵심 설계 패턴이나 알고리즘

대상: {args}
"""
(skills_dir / "explain.md").write_text(explain_skill, encoding="utf-8")

# 스킬 2: 테스트 생성 (fork 모드 — 서브에이전트가 독립적으로 작업)
test_skill = """\
---
name: generate_test
description: 주어진 코드에 대한 테스트를 자동 생성합니다
context: fork
allowed_tools: [Read, Edit, Bash]
when_to_use: 사용자가 테스트 생성을 요청할 때
---
다음 코드에 대한 단위 테스트를 작성해주세요.

테스트 작성 규칙:
- pytest를 사용하세요
- 정상 케이스와 엣지 케이스를 모두 포함하세요
- 각 테스트에 한국어 docstring을 추가하세요
- 테스트 파일은 test_ 접두사를 사용하세요

대상: {args}
"""
(skills_dir / "generate_test.md").write_text(test_skill, encoding="utf-8")

print(f"스킬 디렉토리: {skills_dir}")
print(f"생성된 파일: {[f.name for f in skills_dir.glob('*.md')]}")

# --- 3. 디렉토리에서 스킬 로드 ---
loaded = load_skills_from_dir(skills_dir)
print(f"\n로드된 스킬 수: {len(loaded)}")
for skill in loaded:
    print(f"  [{skill.context:6s}] {skill.name:16s} — {skill.description[:40]}")
    if skill.allowed_tools:
        print(f"           도구 제한: {', '.join(skill.allowed_tools)}")

# --- 4. 레지스트리에 통합 ---
registry = create_default_registry()
registry.register_many(loaded)
print(f"\n통합 레지스트리 스킬 수: {len(registry)}")
print(f"전체 스킬: {registry.list_names()}")

# --- 5. 커스텀 스킬 실행 ---
skill_tool = SkillTool(registry)
result = await skill_tool.call({"skill": "explain", "args": "mini_claude/agent_loop.py"})
print(f"\n=== 커스텀 스킬 'explain' 실행 결과 ===")
print(result.data)

---

## 핵심 설계 원칙 정리

### 1. Skill = Prompt + Tool Restrictions
스킬은 코드가 아니라 프롬프트입니다. "무엇을 하라"는 프롬프트와 "어떤 도구를 써라"는 제한의 조합이 하나의 스킬입니다. 코드 한 줄 없이 마크다운 파일만으로 새 워크플로우를 추가할 수 있습니다.

### 2. inline vs fork — 격리 수준 선택
빠르지만 컨텍스트를 공유하는 inline, 느리지만 독립적인 fork. Claude Code는 작업의 성격에 따라 적절한 실행 모드를 선택합니다. 커밋은 현재 대화 맥락이 필요하므로 inline, 코드 리뷰는 독립적이므로 fork.

### 3. 마크다운 + YAML frontmatter = 선언적 확장
코드를 수정하지 않고도 `.md` 파일을 추가하는 것만으로 에이전트의 능력을 확장할 수 있습니다. 이것이 Claude Code의 `/` 커맨드 시스템의 핵심 설계입니다.

### 4. 프롬프트 엔지니어링이 곧 기능 개발
스킬 시스템에서는 프롬프트를 잘 작성하는 것이 코드를 잘 작성하는 것과 동일합니다. 명확한 지시, 단계별 절차, 도구 제한이 좋은 스킬의 요소입니다.

---

## 다음 Step 미리보기

스킬의 `fork` 모드는 서브에이전트를 생성합니다. 하지만 서브에이전트는 스킬 실행만을 위한 것이 아닙니다.

**Step 11**에서는 Claude Code의 **AgentTool**과 **코디네이터 패턴**을 분석하여:
- **서브에이전트 생성과 컨텍스트 포크** — 부모의 프롬프트 캐시를 공유하는 효율적인 분기
- **내장 에이전트** — explorer, planner, general_purpose의 역할 분담
- **코디네이터 워크플로우** — Research -> Synthesis -> Implementation -> Verification

이 패턴들을 Python으로 구현합니다.